# Reduction Reader Deliverables

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
from pathlib import Path
import subprocess
from astropy.io import fits

# Some setup
COMBINED_DIR = Path("/Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined")
MASTER_DF = pd.DataFrame()

# Helpful Identifiers from .fits:
# h["OBSERVER"]
# h["OBJECT"]
# h["RA"] 
# h["DEC"]

# for asteroids:
# h["SRT_MJD"]
# h["END_MJD"]


# # Important info from .fits
# h["MODE"]
# h["SRT_AM"]
# h["END_AM"]
# h["SRT_TIME"]
# h["END_DATE"]
# h["ORDERS"]

# Moving versus fixed:
# h["YUNITS"]
# h["TC_STDID"]
# h["TC_STDST"]


# First lets loop thru all the availible files in the drive so far!
for file in COMBINED_DIR.iterdir():

    _info_dict = {
        "OBSERVER": None,
        "OBJECT" : None,
        "RA" : None,
        "DEC" : None,
        "SRT_MJD"  : None,
        "END_MJD" : None,
        "MODE" : None,
        "SRT_AM" : None,
        "END_AM" : None,
        "SRT_TIME" : None,
        "END_DATE" : None,
        "ORDER" : None,
        "YUNITS" : None,
        "TC_STDID" : None,
        "TC_STDST" : None
        }

    # For each object we will read in their header and immediately close the file to minimize RAM usage
    hdu = fits.open(f"{file}")
    h = hdu[0].header
    hdu.close()

    # To start, we will distinguish between fixed and moving sources then grab some important info about the data reduction / spectra availible.
    # The main indicator I am using for fixed vs moving objects is the units of the observation: either flux density or reflectance
    # Flux density --> star or galaxy --> fixed object
    # Reflectance --> asteroid --> moving object
    # Will also use tc_type (Telluric correction type) as it also gives reflectance or A0V

    for key in _info_dict.keys():

        try:
            _info_dict[key] = h[key]
        except:
            print(f"Something went wrong with {key} in {file}")

    if h["YUNITS"] == "W m-2 um-1":
        _info_dict["OBJECT_TYPE"] = "Fixed"
        # _info_dict["YUNITS"] = h["YUNITS"]

    elif h["YUNITS"] == "DN s-1" or h["YUNITS"] == "reflectance":
        _info_dict["OBJECT_TYPE"] = "Moving"
        # _info_dict["YUNITS"] = h["YUNITS"]

    # print(_info_dict)

    MASTER_DF = pd.concat((MASTER_DF, pd.DataFrame(_info_dict, index=[0])), ignore_index=True)

Something went wrong with ORDER in /Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-sxd_phi-Per_20090110_2008B083_1-16comb.fits
Something went wrong with ORDER in /Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-prism_J0850+10_20061118_2006B072_233-238comb.fits
Something went wrong with ORDER in /Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-prism_J1254+4346_20100717_2010A999_1-10comb.fits
Something went wrong with ORDER in /Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-sxd_delta-Sco_20090110_2008B083_529-552comb.fits
Something went wrong with ORDER in /Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-sxd_V838-Mon_20090110_2008B083_261-276comb.fits
Something went wrong with ORDER in /Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-prism_Tmove3-id150431_20100715_2010A999_128-133comb.fits
Something went wrong with ORDER in /Users/epritch

,OBSERVER,OBJECT,RA,DEC,SRT_MJD,END_MJD,MODE,SRT_AM,END_AM,SRT_TIME,END_DATE,ORDER,YUNITS,TC_STDID,TC_STDST,OBJECT_TYPE
0,"E Hesselbach, K. Bjorkman",phi Per,+01:43:38.70,++50:41:22.3,54841.342851,54841.34587,ShortXD,1.585,1.608,08:13:42.334733,2009-01-10,None,W m-2 um-1,HD12365,A0V,Fixed
1,Your_Name,2mass 0850+10: L/T binary,08:50:35.28,+10:57:15.5,54057.612701,54057.624472,LowRes15,1.025,1.016,14:42:17.386584,2006-11-18,None,W m-2 um-1,hd 74721,A0V,Fixed
2,Your_Name,redred-id80435,+12:54:38.19,++43:46:54.2,55394.240318,55394.255071,LowRes15,1.24,1.299,05:46:03.469897,2010-07-17,None,W m-2 um-1,HD 109615,A0V,Fixed
3,"E Hesselbach, K. Bjorkman",delta Sco,+16:00:20.46,-22:37:19.5,54841.637731,54841.638719,ShortXD,2.777,2.744,15:18:19.967282,2009-01-10,None,W m-2 um-1,HD 131951,A0V,Fixed
4,"E Hesselbach, K. Bjorkman",V838 Mon,+07:04:04.47,-03:50:53.9,54841.469131,54841.482366,ShortXD,1.147,1.184,11:15:32.942429,2009-01-10,None,W m-2 um-1,HD 50931,A0V,Fixed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,"Vishnu Reddy, Will Swearson",423 Diotima,+04:27:28.78,++17:18:08.3,55088.535633,55088.593854,LowRes15,1.201,1.031,12:51:18.722121,2009-09-14,None,DN s-1,SAO 93936,G2+V,Moving
535,Looper,TWA 30B,+11:32:17.96,-30:18:23.2,55223.634022,55223.651826,LowRes15,1.808,1.971,15:12:59.466583,2010-01-27,None,DN s-1,HD 98949,A0V,Moving
536,Your_Name,bPic13774,+22:17:16.77,++23:09:26.1,55394.645792,55394.646745,ShortXD,1.23,1.234,15:29:56.464737,2010-07-17,None,W m-2 um-1,HD 212643,A0V,Fixed
537,"E Hesselbach, K. Bjorkman",nu Gem,+06:28:57.19,++20:12:43.1,54841.499811,54841.504701,ShortXD,1.219,1.245,11:59:43.651338,2009-01-10,None,W m-2 um-1,HD 43583,A0V,Fixed
